In [ ]:
import pandas as pd
import numpy as np
import os, sys
from tqdm.notebook import tqdm
from pathlib import Path
from plotly import express as px

In [ ]:
class Estimator:
    def __init__(self):
        pass

    def fit(self):
        pass

    def __call__(self, well_data:pd.DataFrame, typewell_data:pd.DataFrame)->pd.Series:
        """Returns a TVT pd.Series"""
        pass

class Evaluator:
    def __init__(self):
        pass

    def evaluate_estimator(self, estimator:Estimator, file_path:Path=Path("rogii-wellbore-geology-prediction/train",)):
        well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }
        tvt_trues, tvt_preds= {}, {}
        for fname, paths in tqdm(well_files.items()):
            well_data, typewell_data = pd.read_csv(paths["Well"]), pd.read_csv(paths["TypeWell"])
            first_pred_idx = well_data.TVT_input.shift(1).dropna().index[-1]
            tvt_trues[fname] = well_data.loc[first_pred_idx:,"TVT"]
            tvt_preds[fname] = estimator(well_data)
        tvt_trues, tvt_preds = pd.DataFrame(tvt_trues), pd.DataFrame(tvt_preds)
        tvt_errors = tvt_preds - tvt_trues
        rmse = np.nanmean(tvt_errors.values.flatten()**2)**0.5
        return tvt_trues, tvt_preds, tvt_errors, rmse
    
    def make_submission(self, estimator:Estimator, file_path:Path=Path("rogii-wellbore-geology-prediction/test"),submission_file="submission.csv"):
        well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }
        tvt_preds = []
        for fname, paths in tqdm(well_files.items()):
            well_data, typewell_data = pd.read_csv(paths["Well"]), pd.read_csv(paths["TypeWell"])
            tvt_pred = estimator(well_data).rename("tvt")
            tvt_pred.index = pd.Series(fname + "_"+tvt_pred.index.values.astype(str),name="id")
            tvt_preds.append(tvt_pred)
        tvt_preds = pd.concat(tvt_preds,axis=0)
        tvt_preds.to_csv(submission_file)
        

In [ ]:
class NaiveEstimator(Estimator):
    def __call__(self, well_data:pd.DataFrame, typewell_data:pd.DataFrame=None)->pd.Series:
        first_pred_idx = well_data.TVT_input.shift(1).dropna().index[-1]
        last_tvt = well_data.TVT_input.dropna().iloc[-1]
        pred_tvt = pd.Series(last_tvt,index=well_data.index[first_pred_idx:])
        return pred_tvt
        
        

In [ ]:
evaluator = Evaluator()
estimator = NaiveEstimator()

In [ ]:
tvt_trues, tvt_preds, tvt_errors, rmse = evaluator.evaluate_estimator(estimator)

  0%|          | 0/773 [00:00<?, ?it/s]

In [ ]:
rmse

In [ ]:
evaluator.make_submission(estimator)

  0%|          | 0/3 [00:00<?, ?it/s]